## Model Eval and performance measures

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === Define your model class ===
class MiniRocketMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MiniRocketMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(0.3)
        self.out = nn.Linear(64, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        return self.out(x)

# === Load model input dimension and class labels ===
input_dim = np.load("models/v3/minirocket_input_dim.npy")[0]
class_names = np.load("models/v3/y_labels.npy", allow_pickle=True)

# === Load model state ===
model = MiniRocketMLP(input_dim=input_dim, num_classes=len(class_names))
model.load_state_dict(torch.load("models/v3/Mini-RocketMLP.pt", map_location=torch.device("cpu")))
model.eval()

# === Load dataset (already transformed with MiniRocket) ===
df = pd.read_csv("Data/New/processed/cleaned_MAPB.csv")
feature_cols = [col for col in df.columns if col not in ['bond_type', 'drug_name'] and not col.startswith("meta")]
X = df[feature_cols].values.astype(np.float32)

# Sanity check
if X.shape[1] != input_dim:
    raise ValueError(f"Expected input dim {input_dim}, but got {X.shape[1]}")

# === Run predictions ===
X_tensor = torch.from_numpy(X)
with torch.no_grad():
    logits = model(X_tensor)
    probs = F.softmax(logits, dim=1).numpy()  # shape: (N, num_classes)

# === Plot class-wise confidence ===
frames = np.arange(len(probs))
plt.figure(figsize=(12, 6))
for class_idx in range(len(class_names)):
    plt.plot(frames, probs[:, class_idx], label=f'{class_names[class_idx]}', alpha=0.8)

plt.xlabel("Frame Index")
plt.ylabel("Model Confidence")
plt.title("Model Confidence per Class across Frames")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


ValueError: could not convert string to float: 'Hydrophobic'